## 01 - Neural Classification - Pitt
In this notebook, we will do a neural architecture with the Pitt dataset.
This model contains:
- Bert embeddings (using sentence-transformer library)
- Neural (Multi-layer perceptron) classifier from Sk-learn

In [1]:
# !pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

In [3]:
embed_model_name = "LaBSe"

In [4]:
data_path = "../../../data/" 

# please point to correct downloaded transcribed folders of the Pitt dataset
train_fname = data_path + "pitt/pitt_train_txt.csv"
dev_fname = data_path + "pitt/pitt_dev_txt.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 337
Dev df = 115


In [5]:
train_df.head()

,file_id,text,target,id,split_name
0,003-0,here's a cookie jar .\nand the lid is off the ...,AD,3,train
1,005-0,the boy is slipping off the stool .\nhe's tryi...,AD,5,train
2,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD,5,train
3,007-1,xxx .\ncan I look at it and tell you ?\noh oka...,AD,7,train
4,007-3,well the girl is telling the boy to get the co...,AD,7,train


In [6]:
dev_df.head()

,file_id,text,target,id,split_name
0,016-0,guy or a boy reaching for a cookie jar &-uh ti...,AD,16,dev
1,016-1,uh the water's spilling over .\ndrying the dis...,AD,16,dev
2,016-3,a boy is reaching for the cookie jar and tripp...,AD,16,dev
3,016-4,well the little girl wants the cookie that the...,AD,16,dev
4,033-0,okay .\napparently a mother and a daughter and...,AD,33,dev


### Pre-processing
We will lowercase the text for better efficiency

In [7]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [8]:
vectorizer = SentenceTransformer(embed_model_name) 

C:\Users\Minh UBC\miniconda3\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [9]:
corpus = train_df["text_lower"]
print("Creating text embeddings...")
X_train = vectorizer.encode(corpus)
print("Completed")

Creating text embeddings...
Completed


In [10]:
print(X_train.shape)

(337, 768)


### Train Neural Classifier

We can now train a neural logistic regression classifier using processed X_train

In [11]:
y_train = train_df["target"]

print("Training neural classifier...")
hidden_size = 500
clf = MLPClassifier(solver="lbfgs",
                    alpha=1e-5,
                    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
                    learning_rate_init=0.0001,
                    max_iter=500, 
                    random_state=42
                   )

clf.fit(X_train, y_train)
print("Completed")

Training neural classifier...
Completed


In [12]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 100.00 %


### Get accuracy using cross validation

In [13]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=5)
print("Completed")
print(scores)

Performing cross-validation...
Completed
[0.58823529 0.70588235 0.68656716 0.71641791 0.70149254]


In [14]:
print("Average =", np.average(scores))

Average = 0.6797190517998244


In [15]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

          AD       1.00      1.00      1.00       187
          CN       1.00      1.00      1.00       150

    accuracy                           1.00       337
   macro avg       1.00      1.00      1.00       337
weighted avg       1.00      1.00      1.00       337



### Now we can evaluate on dev set

In [16]:
X_dev = vectorizer.encode(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (115, 768)
y_dev len = 115


In [17]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 73.04 %


In [18]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred))

              precision    recall  f1-score   support

          AD       0.76      0.75      0.76        64
          CN       0.69      0.71      0.70        51

    accuracy                           0.73       115
   macro avg       0.73      0.73      0.73       115
weighted avg       0.73      0.73      0.73       115



### Process prediction for test set

In [19]:
test_fname = data_path + "pitt/pitt_test_txt.csv"

test_df = pd.read_csv(test_fname)
print("Test df =", len(test_df))
test_df["text_lower"] = test_df["text"].apply(lambda text: text.lower().strip())
test_df.head()

Test df = 97


,file_id,text,target,id,split_name,text_lower
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,1,test,mhm .\nalright .\nthere's &-um a young boy tha...
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,1,test,mhm .\nthere's a young boy &-uh going in a coo...
2,023-0,(..) well I see the mother .\nwhat's she doin(...,AD,23,test,(..) well i see the mother .\nwhat's she doin(...
3,023-2,well you see <the little> the little boy stea...,AD,23,test,well you see <the little> the little boy stea...
4,039-0,okay we see &-um first of all &-uh the sink is...,AD,39,test,okay we see &-um first of all &-uh the sink is...


In [20]:
# create X_test embeddings
print("Creating text embeddings...")
X_test = vectorizer.encode(test_df["text_lower"])
print("Completed")

Creating text embeddings...
Completed


In [21]:
test_df["prediction"] = clf.predict(X_test)
test_df.head()

,file_id,text,target,id,split_name,text_lower,prediction
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD,1,test,mhm .\nalright .\nthere's &-um a young boy tha...,AD
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD,1,test,mhm .\nthere's a young boy &-uh going in a coo...,AD
2,023-0,(..) well I see the mother .\nwhat's she doin(...,AD,23,test,(..) well i see the mother .\nwhat's she doin(...,AD
3,023-2,well you see <the little> the little boy stea...,AD,23,test,well you see <the little> the little boy stea...,AD
4,039-0,okay we see &-um first of all &-uh the sink is...,AD,39,test,okay we see &-um first of all &-uh the sink is...,AD


In [22]:
correct_count = len(test_df.query("target == prediction"))
print(f"Correct count = {correct_count}, accuracy = {correct_count*100/len(test_df):.0f} %")

Correct count = 70, accuracy = 72 %


In [23]:
print(classification_report(test_df["target"], test_df["prediction"]))

              precision    recall  f1-score   support

          AD       0.71      0.85      0.78        55
          CN       0.74      0.55      0.63        42

    accuracy                           0.72        97
   macro avg       0.73      0.70      0.70        97
weighted avg       0.73      0.72      0.71        97

